In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

folder_path = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Video/Attack/replay_attack_videos"

renamed = 0

for file in os.listdir(folder_path):
    if "-PA" in file:
        old_path = os.path.join(folder_path, file)

        new_name = file.replace("-PA", "-RA")
        new_path = os.path.join(folder_path, new_name)

        os.rename(old_path, new_path)
        renamed += 1

        print(f"Renamed: {file} → {new_name}")

print("\nTotal files renamed:", renamed)

Renamed: supun-kanakarathna-1770217531977-wk28pni9c7d-video-PA.webm.webm → supun-kanakarathna-1770217531977-wk28pni9c7d-video-RA.webm.webm
Renamed: vijayakalyani-balajeyandhan-1771049545786-xcfsda9eado-video-PA.webm → vijayakalyani-balajeyandhan-1771049545786-xcfsda9eado-video-RA.webm
Renamed: 1770003685383-va6cgq2e3pc-video-PA.webm → 1770003685383-va6cgq2e3pc-video-RA.webm
Renamed: pradeep-periyasamy-1770301371712-34a8ac6kmjn-video-PA.webm → pradeep-periyasamy-1770301371712-34a8ac6kmjn-video-RA.webm
Renamed: yuresh-anjana-1770218434339-0208d18knkyy-video-PA.webm → yuresh-anjana-1770218434339-0208d18knkyy-video-RA.webm
Renamed: 1770028923875-1hygdlyttxt-video-PA.webm → 1770028923875-1hygdlyttxt-video-RA.webm
Renamed: 1770008603482-ya1pblgtah-video 11-PA.webm → 1770008603482-ya1pblgtah-video 11-RA.webm
Renamed: 1770016503680-7cdim9wd80u-video-PA.webm → 1770016503680-7cdim9wd80u-video-RA.webm
Renamed: 1770087965227-1baaery53s3i-video-PA.webm → 1770087965227-1baaery53s3i-video-RA.webm
Ren

In [4]:
import os
import pandas as pd

# 🔹 Folder where your replay (RA) videos are
replay_folder = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Video/Attack/replay_attack_videos"

video_extensions = (".mp4", ".mov", ".webm", ".avi", ".mkv")

rows = []

for file in os.listdir(replay_folder):
    if file.lower().endswith(video_extensions):
        full_path = os.path.join(replay_folder, file)

        # 🔹 Since filenames DON'T contain participant ID,
        # we generate a unique ID per file
        participant_id = os.path.splitext(file)[0]

        rows.append({
            "participant_id": participant_id,
            "source_identity": participant_id,   # replay = same identity
            "attack_type": "replay",
            "file_path": full_path,
            "label": 1
        })

replay_df = pd.DataFrame(rows)

print("Total replay videos:", len(replay_df))
print(replay_df.head())

Total replay videos: 119
                                      participant_id  \
0  supun-kanakarathna-1770217531977-wk28pni9c7d-v...   
1  vijayakalyani-balajeyandhan-1771049545786-xcfs...   
2                 1770003685383-va6cgq2e3pc-video-RA   
3  pradeep-periyasamy-1770301371712-34a8ac6kmjn-v...   
4  yuresh-anjana-1770218434339-0208d18knkyy-video-RA   

                                     source_identity attack_type  \
0  supun-kanakarathna-1770217531977-wk28pni9c7d-v...      replay   
1  vijayakalyani-balajeyandhan-1771049545786-xcfs...      replay   
2                 1770003685383-va6cgq2e3pc-video-RA      replay   
3  pradeep-periyasamy-1770301371712-34a8ac6kmjn-v...      replay   
4  yuresh-anjana-1770218434339-0208d18knkyy-video-RA      replay   

                                           file_path  label  
0  /content/drive/MyDrive/Final_Dataset/Sam_final...      1  
1  /content/drive/MyDrive/Final_Dataset/Sam_final...      1  
2  /content/drive/MyDrive/Final_Dataset/Sam

In [5]:
missing = replay_df["file_path"].apply(lambda x: not os.path.exists(x)).sum()
print("Missing files:", missing)

Missing files: 0


In [6]:
save_path = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Video/replay_attack_metadata.csv"
replay_df.to_csv(save_path, index=False)

print("Saved to:", save_path)

Saved to: /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Video/replay_attack_metadata.csv


In [7]:
import pandas as pd
import os

base_path = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Video/"

# Load each metadata file
genuine_df = pd.read_csv(base_path + "genuine_face_metadata.csv")
replay_df = pd.read_csv(base_path + "replay_attack_metadata.csv")
imp_df = pd.read_csv(base_path + "impersonation_attack_metadata.csv")
logical_df = pd.read_csv(base_path + "logical_attack_metadata.csv")

# Keep only the required columns
required_cols = ["participant_id", "source_identity", "attack_type", "file_path", "label"]

genuine_df = genuine_df[required_cols].copy()
replay_df = replay_df[required_cols].copy()
imp_df = imp_df[required_cols].copy()
logical_df = logical_df[required_cols].copy()

# Combine
final_face_df = pd.concat(
    [genuine_df, replay_df, imp_df, logical_df],
    ignore_index=True
)

# Basic cleanup
final_face_df = final_face_df.dropna(subset=["file_path", "label"]).reset_index(drop=True)
final_face_df["label"] = final_face_df["label"].astype(int)

# Check file existence
final_face_df["exists"] = final_face_df["file_path"].apply(os.path.exists)

print("Total rows:", len(final_face_df))
print("\nAttack distribution:")
print(final_face_df["attack_type"].value_counts())

print("\nLabel distribution:")
print(final_face_df["label"].value_counts())

print("\nMissing files:", (~final_face_df["exists"]).sum())

# Keep only valid files
final_face_df = final_face_df[final_face_df["exists"]].drop(columns=["exists"]).reset_index(drop=True)

print("\nFinal usable rows:", len(final_face_df))
print(final_face_df.head())

# Save final dataset
save_path = base_path + "final_face_dataset.csv"
final_face_df.to_csv(save_path, index=False)

print("\nSaved:", save_path)

Total rows: 434

Attack distribution:
attack_type
genuine          119
replay           119
impersonation     98
logical           98
Name: count, dtype: int64

Label distribution:
label
1    315
0    119
Name: count, dtype: int64

Missing files: 217

Final usable rows: 217
                                      participant_id  \
0  supun-kanakarathna-1770217531977-wk28pni9c7d-v...   
1  vijayakalyani-balajeyandhan-1771049545786-xcfs...   
2                 1770003685383-va6cgq2e3pc-video-RA   
3  pradeep-periyasamy-1770301371712-34a8ac6kmjn-v...   
4  yuresh-anjana-1770218434339-0208d18knkyy-video-RA   

                                     source_identity attack_type  \
0  supun-kanakarathna-1770217531977-wk28pni9c7d-v...      replay   
1  vijayakalyani-balajeyandhan-1771049545786-xcfs...      replay   
2                 1770003685383-va6cgq2e3pc-video-RA      replay   
3  pradeep-periyasamy-1770301371712-34a8ac6kmjn-v...      replay   
4  yuresh-anjana-1770218434339-0208d18knkyy-vide

In [8]:
import pandas as pd

inv_df = pd.read_csv("/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Video/FINAL_VIDEO_INVENTORY.csv")

print(inv_df.columns.tolist())
print(inv_df.head())
print(inv_df.shape)

['session_token', 'video_filename', 'video_quality', 'full_path', 'Gender', 'participant_id']
               session_token  \
0  1769877813423-fzmq80fcxer   
1  1771049545786-xcfsda9eado   
2   1770445787080-lqw56ag66w   
3  1770259504525-yq212ny4ozl   
4   1770445863400-3hz9dnvpov   

                                      video_filename   video_quality  \
0               1769877813423-fzmq80fcxer-video.webm  perfect videos   
1  vijayakalyani-balajeyandhan-1771049545786-xcfs...  perfect videos   
2          umair-1770445787080-lqw56ag66w-video.webm  perfect videos   
3  uditha-karawita-1770259504525-yq212ny4ozl-vide...  perfect videos   
4  thrishonika-sivarajah-1770445863400-3hz9dnvpov...  perfect videos   

                                           full_path Gender participant_id  
0  /content/drive/MyDrive/Final Dataset/Sam - fin...      M           P001  
1  /content/drive/MyDrive/Final Dataset/Sam - fin...      F           P002  
2  /content/drive/MyDrive/Final Dataset/Sam - fin

In [11]:
import os
import pandas as pd

# Paths
inventory_path = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Video/FINAL_VIDEO_INVENTORY.csv"
replay_folder = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Video/Attack/replay_attack_videos"
save_path = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Video/replay_attack_metadata_fixed.csv"

video_extensions = (".mp4", ".mov", ".webm", ".avi", ".mkv")

# Load genuine inventory
inv_df = pd.read_csv(inventory_path).copy()

# Keep only needed columns
inv_df = inv_df[["video_filename", "full_path", "participant_id"]].copy()

# Normalize inventory filename keys
def normalize_name(name):
    name = str(name).strip()
    name = os.path.basename(name)
    name = os.path.splitext(name)[0]
    return name.lower()

inv_df["inventory_key"] = inv_df["video_filename"].apply(normalize_name)

# Build lookup: original genuine filename -> participant_id
inventory_lookup = dict(zip(inv_df["inventory_key"], inv_df["participant_id"]))

rows = []
unmatched = []

for file in os.listdir(replay_folder):
    if not file.lower().endswith(video_extensions):
        continue

    full_path = os.path.join(replay_folder, file)

    # normalize replay filename
    replay_key = normalize_name(file)

    # strip replay suffixes
    # handles names like:
    # xxx-video-RA.webm
    # xxx-RA.webm
    original_key = replay_key.replace("-ra", "")
    original_key = original_key.replace("_ra", "")

    # cleanup accidental double separators if any
    original_key = original_key.replace("--", "-").replace("__", "_").strip("-_")

    participant_id = inventory_lookup.get(original_key)

    if participant_id is None:
        unmatched.append({
            "replay_file": file,
            "derived_original_key": original_key
        })
        continue

    rows.append({
        "participant_id": participant_id,
        "source_identity": participant_id,   # replay = same identity, but spoofed presentation
        "attack_type": "replay",
        "file_path": full_path,
        "label": 1
    })

replay_df = pd.DataFrame(rows)
unmatched_df = pd.DataFrame(unmatched)

print("Matched replay samples:", len(replay_df))
print("Unmatched replay samples:", len(unmatched_df))

if len(replay_df) > 0:
    print("\nReplay metadata preview:")
    print(replay_df.head())

if len(unmatched_df) > 0:
    print("\nUnmatched preview:")
    print(unmatched_df.head())

# Save results
replay_df.to_csv(save_path, index=False)
print("\nSaved fixed replay metadata to:", save_path)

if len(unmatched_df) > 0:
    unmatched_save = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Video/replay_unmatched_debug.csv"
    unmatched_df.to_csv(unmatched_save, index=False)
    print("Saved unmatched debug file to:", unmatched_save)

Matched replay samples: 114
Unmatched replay samples: 5

Replay metadata preview:
  participant_id source_identity attack_type  \
0           P002            P002      replay   
1           P052            P052      replay   
2           P094            P094      replay   
3           P096            P096      replay   
4           P047            P047      replay   

                                           file_path  label  
0  /content/drive/MyDrive/Final_Dataset/Sam_final...      1  
1  /content/drive/MyDrive/Final_Dataset/Sam_final...      1  
2  /content/drive/MyDrive/Final_Dataset/Sam_final...      1  
3  /content/drive/MyDrive/Final_Dataset/Sam_final...      1  
4  /content/drive/MyDrive/Final_Dataset/Sam_final...      1  

Unmatched preview:
                                         replay_file  \
0  supun-kanakarathna-1770217531977-wk28pni9c7d-v...   
1  dwayne-ramsay-1770725471062-pvz0nwke0cp-video-...   
2  pragash-sasitharan-1770557958937-o7zm1xx7z5-vi...   
3            

In [ ]:
import os
import pandas as pd

# 🔹 Load your current genuine metadata
genuine_path = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Video/genuine_face_metadata.csv"
genuine_df = pd.read_csv(genuine_path)

# 🔹 Correct base folder
genuine_folder = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Video/Genuine_Folder"

# 🔹 Build filename → full path mapping
file_lookup = {}

for root, _, files in os.walk(genuine_folder):
    for file in files:
        file_lookup[file] = os.path.join(root, file)

print("Total files found in Genuine_Folder:", len(file_lookup))

# 🔹 Function to fix path
def fix_path(old_path):
    filename = os.path.basename(old_path)
    return file_lookup.get(filename, None)

# 🔹 Apply fix
genuine_df["file_path"] = genuine_df["file_path"].apply(fix_path)

# 🔹 Check missing
missing = genuine_df["file_path"].isna().sum()
print("Missing after fix:", missing)

# 🔹 Drop rows that couldn't be matched
genuine_df = genuine_df.dropna(subset=["file_path"]).reset_index(drop=True)

print("Final rows after cleaning:", len(genuine_df))

# 🔹 Save fixed version
save_path = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Video/genuine_face_metadata_fixed.csv"
genuine_df.to_csv(save_path, index=False)

print("Saved fixed genuine metadata:", save_path)

In [12]:
import os
import pandas as pd

# 🔹 Load metadata
genuine_path = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Video/genuine_face_metadata.csv"
genuine_df = pd.read_csv(genuine_path)

# 🔹 Root folder (this contains all subfolders you showed)
genuine_root = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Video/Genuine_Folder"

# 🔹 Build full lookup (recursively through ALL subfolders)
file_lookup = {}

for root, _, files in os.walk(genuine_root):
    for file in files:
        # if duplicates exist, keep first (safe assumption)
        if file not in file_lookup:
            file_lookup[file] = os.path.join(root, file)

print("Total genuine videos found:", len(file_lookup))

Total genuine videos found: 119


In [13]:
def fix_path(old_path):
    filename = os.path.basename(old_path)
    return file_lookup.get(filename, None)

genuine_df["file_path"] = genuine_df["file_path"].apply(fix_path)

In [14]:
missing = genuine_df["file_path"].isna().sum()
print("Missing after fix:", missing)

Missing after fix: 1


In [15]:
missing_rows = genuine_df[genuine_df["file_path"].isna()]
print(missing_rows.head())

    participant_id source_identity attack_type file_path  label
117           P074            P074     genuine      None      0


In [16]:
genuine_df = genuine_df.dropna(subset=["file_path"]).reset_index(drop=True)

save_path = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Video/genuine_face_metadata_fixed.csv"
genuine_df.to_csv(save_path, index=False)

print("Saved fixed file:", save_path)
print("Final rows:", len(genuine_df))

Saved fixed file: /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Video/genuine_face_metadata_fixed.csv
Final rows: 118


In [17]:
import pandas as pd

# Load the combined genuine + impersonation dataset
df = pd.read_csv("/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Video/final_face_impersonation_dataset.csv")

# Keep only impersonation rows
imp_df = df[df["attack_type"] == "impersonation"].copy()

# Optional sanity checks
print("Impersonation rows:", len(imp_df))
print(imp_df.head())
print(imp_df["label"].value_counts())
print("Rows where participant_id == source_identity:", (imp_df["participant_id"] == imp_df["source_identity"]).sum())

# Save standalone impersonation metadata
save_path = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Video/impersonation_attack_metadata_fixed.csv"
imp_df.to_csv(save_path, index=False)

print("Saved:", save_path)

Impersonation rows: 98
    participant_id source_identity    attack_type  \
119           P001            P003  impersonation   
120           P003            P004  impersonation   
121           P004            P006  impersonation   
122           P006            P007  impersonation   
123           P007            P008  impersonation   

                                             file_path  label  
119  /content/drive/MyDrive/Final Dataset/Sam - fin...      1  
120  /content/drive/MyDrive/Final Dataset/Sam - fin...      1  
121  /content/drive/MyDrive/Final Dataset/Sam - fin...      1  
122  /content/drive/MyDrive/Final Dataset/Sam - fin...      1  
123  /content/drive/MyDrive/Final Dataset/Sam - fin...      1  
label
1    98
Name: count, dtype: int64
Rows where participant_id == source_identity: 0
Saved: /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Video/impersonation_attack_metadata_fixed.csv


In [18]:
import pandas as pd
import os

base_path = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Video/"

# 🔹 Load files (use FIXED ones where applicable)
genuine_df = pd.read_csv(base_path + "genuine_face_metadata_fixed.csv")
replay_df = pd.read_csv(base_path + "replay_attack_metadata_fixed.csv")
imp_df = pd.read_csv(base_path + "impersonation_attack_metadata_fixed.csv")
logical_df = pd.read_csv(base_path + "logical_attack_metadata.csv")

# 🔹 Required columns
required_cols = ["participant_id", "source_identity", "attack_type", "file_path", "label"]

genuine_df = genuine_df[required_cols].copy()
replay_df = replay_df[required_cols].copy()
imp_df = imp_df[required_cols].copy()
logical_df = logical_df[required_cols].copy()

# 🔹 Combine all
final_face_df = pd.concat(
    [genuine_df, replay_df, imp_df, logical_df],
    ignore_index=True
)

# 🔹 Clean
final_face_df = final_face_df.dropna(subset=["file_path", "label"]).reset_index(drop=True)
final_face_df["label"] = final_face_df["label"].astype(int)

# 🔍 Check file existence
final_face_df["exists"] = final_face_df["file_path"].apply(os.path.exists)

print("Total rows BEFORE cleaning:", len(final_face_df))

print("\nAttack distribution:")
print(final_face_df["attack_type"].value_counts())

print("\nLabel distribution:")
print(final_face_df["label"].value_counts())

print("\nMissing files:", (~final_face_df["exists"]).sum())

# 🔥 Keep only valid paths
final_face_df = final_face_df[final_face_df["exists"]].drop(columns=["exists"]).reset_index(drop=True)

print("\nFinal usable rows:", len(final_face_df))
print(final_face_df.head())

Total rows BEFORE cleaning: 428

Attack distribution:
attack_type
genuine          118
replay           114
impersonation     98
logical           98
Name: count, dtype: int64

Label distribution:
label
1    310
0    118
Name: count, dtype: int64

Missing files: 98

Final usable rows: 330
  participant_id source_identity attack_type  \
0           P001            P001     genuine   
1           P002            P002     genuine   
2           P003            P003     genuine   
3           P004            P004     genuine   
4           P005            P005     genuine   

                                           file_path  label  
0  /content/drive/MyDrive/Final_Dataset/Sam_final...      0  
1  /content/drive/MyDrive/Final_Dataset/Sam_final...      0  
2  /content/drive/MyDrive/Final_Dataset/Sam_final...      0  
3  /content/drive/MyDrive/Final_Dataset/Sam_final...      0  
4  /content/drive/MyDrive/Final_Dataset/Sam_final...      0  


In [19]:
save_path = base_path + "final_face_dataset.csv"
final_face_df.to_csv(save_path, index=False)

print("Saved final dataset:", save_path)

Saved final dataset: /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Video/final_face_dataset.csv


In [2]:
import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, accuracy_score

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: Tesla T4


load final face dataset

In [4]:
df = pd.read_csv("/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Video/final_face_dataset.csv")

print("Shape:", df.shape)
print(df.head())
print("\nAttack distribution:")
print(df["attack_type"].value_counts())
print("\nLabel distribution:")
print(df["label"].value_counts())

Shape: (330, 5)
  participant_id source_identity attack_type  \
0           P001            P001     genuine   
1           P002            P002     genuine   
2           P003            P003     genuine   
3           P004            P004     genuine   
4           P005            P005     genuine   

                                           file_path  label  
0  /content/drive/MyDrive/Final_Dataset/Sam_final...      0  
1  /content/drive/MyDrive/Final_Dataset/Sam_final...      0  
2  /content/drive/MyDrive/Final_Dataset/Sam_final...      0  
3  /content/drive/MyDrive/Final_Dataset/Sam_final...      0  
4  /content/drive/MyDrive/Final_Dataset/Sam_final...      0  

Attack distribution:
attack_type
genuine    118
replay     114
logical     98
Name: count, dtype: int64

Label distribution:
label
1    212
0    118
Name: count, dtype: int64


keep only valid files

In [5]:
df = df.dropna(subset=["file_path", "label"]).copy()
df["label"] = df["label"].astype(int)

df["exists"] = df["file_path"].apply(os.path.exists)
print("Missing files:", (~df["exists"]).sum())

df = df[df["exists"]].drop(columns=["exists"]).reset_index(drop=True)
print("Final usable samples:", len(df))

Missing files: 0
Final usable samples: 330


Train/Validation Split

In [6]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("\nTrain labels:")
print(train_df["label"].value_counts())
print("\nVal labels:")
print(val_df["label"].value_counts())

Train shape: (264, 5)
Val shape: (66, 5)

Train labels:
label
1    170
0     94
Name: count, dtype: int64

Val labels:
label
1    42
0    24
Name: count, dtype: int64


Memory-safe dataset


In [7]:
class LazyFaceFrameDataset(Dataset):
  def __init__(self, meta_df, transform=None, num_frames=4):
        self.transform = transform
        self.items = []

        for _, row in meta_df.iterrows():
            video_path = row["file_path"]
            label = int(row["label"])
            participant_id = row.get("participant_id", None)
            source_identity = row.get("source_identity", None)
            attack_type = row.get("attack_type", None)

            cap = cv2.VideoCapture(video_path)
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            cap.release()

            if total_frames <= 0:
                continue

            frame_indices = np.linspace(
                0,
                total_frames - 1,
                num=min(num_frames, total_frames),
                dtype=int
            )

            for frame_idx in frame_indices:
                self.items.append({
                    "video_path": video_path,
                    "frame_idx": int(frame_idx),
                    "label": label,
                    "participant_id": participant_id,
                    "source_identity": source_identity,
                    "attack_type": attack_type
                })
  def __len__(self):
        return len(self.items)

  def __getitem__(self, idx):
        item = self.items[idx]

        cap = cv2.VideoCapture(item["video_path"])
        cap.set(cv2.CAP_PROP_POS_FRAMES, item["frame_idx"])
        ret, frame = cap.read()
        cap.release()

        if not ret:
            frame = np.zeros((224, 224, 3), dtype=np.uint8)
        else:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        img = Image.fromarray(frame)

        if self.transform:
            img = self.transform(img)

        return (
            img,
            item["label"],
            item["video_path"],
            item["participant_id"],
            item["source_identity"],
            item["attack_type"]
        )

Image transforms

In [8]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

clear memory

In [10]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()
print("Memory cleared")

Memory cleared


build datasets & loaders

In [11]:
train_dataset = LazyFaceFrameDataset(train_df, transform=train_transform, num_frames=4)
val_dataset = LazyFaceFrameDataset(val_df, transform=val_transform, num_frames=4)

print("\nTrain frames:", len(train_dataset))
print("Val frames:", len(val_dataset))

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0)


Train frames: 1020
Val frames: 256


Build model

In [12]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

print("\nModel ready.")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 172MB/s]



Model ready.


Loss and optimizer

In [13]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

Train / validate functions

In [4]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels, _, _, _, _ in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for images, labels, _, _, _, _ in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = outputs.argmax(dim=1)

            running_loss += loss.item() * images.size(0)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    auc = roc_auc_score(all_labels, all_probs)
    return running_loss / total, correct / total, auc, all_labels, all_preds, all_probs

Train

In [15]:
num_epochs = 5
best_val_auc = 0.0
best_model_path = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Video/face_binary_resnet18_best.pth"

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_auc, all_labels, all_preds, all_probs = validate_one_epoch(
        model, val_loader, criterion, device
    )

    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f}")

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), best_model_path)
        print("Saved best model.")


Epoch 1/5
Train Loss: 0.4581 | Train Acc: 0.7990
Val   Loss: 0.5105 | Val   Acc: 0.7891 | Val AUC: 0.8353
Saved best model.

Epoch 2/5
Train Loss: 0.3209 | Train Acc: 0.8657
Val   Loss: 0.3723 | Val   Acc: 0.8320 | Val AUC: 0.9110
Saved best model.

Epoch 3/5
Train Loss: 0.2457 | Train Acc: 0.9069
Val   Loss: 0.3360 | Val   Acc: 0.8828 | Val AUC: 0.9308
Saved best model.

Epoch 4/5
Train Loss: 0.2260 | Train Acc: 0.9225
Val   Loss: 0.3494 | Val   Acc: 0.8594 | Val AUC: 0.9110

Epoch 5/5
Train Loss: 0.2018 | Train Acc: 0.9186
Val   Loss: 0.3060 | Val   Acc: 0.8594 | Val AUC: 0.9368
Saved best model.


redoing it since after building the model, the GPU stopped.

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
import os
import gc
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix,
    accuracy_score
)

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cpu


In [3]:
df = pd.read_csv("/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Video/final_face_dataset.csv")

df = df.dropna(subset=["file_path", "label"]).copy()
df["label"] = df["label"].astype(int)

df["exists"] = df["file_path"].apply(os.path.exists)
print("Missing files:", (~df["exists"]).sum())

df = df[df["exists"]].drop(columns=["exists"]).reset_index(drop=True)

print("Usable samples:", len(df))
print(df["attack_type"].value_counts())
print(df["label"].value_counts())

Missing files: 0
Usable samples: 330
attack_type
genuine    118
replay     114
logical     98
Name: count, dtype: int64
label
1    212
0    118
Name: count, dtype: int64


recreating same train/val split

In [4]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)

Train shape: (264, 5)
Val shape: (66, 5)


rebuilding memory safe dataset

In [5]:
class LazyFaceFrameDataset(Dataset):
    def __init__(self, meta_df, transform=None, num_frames=4):
        self.transform = transform
        self.items = []

        for _, row in meta_df.iterrows():
            video_path = row["file_path"]
            label = int(row["label"])
            participant_id = row.get("participant_id", None)
            source_identity = row.get("source_identity", None)
            attack_type = row.get("attack_type", None)

            cap = cv2.VideoCapture(video_path)
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            cap.release()

            if total_frames <= 0:
                continue

            frame_indices = np.linspace(
                0,
                total_frames - 1,
                num=min(num_frames, total_frames),
                dtype=int
            )

            for frame_idx in frame_indices:
                self.items.append({
                    "video_path": video_path,
                    "frame_idx": int(frame_idx),
                    "label": label,
                    "participant_id": participant_id,
                    "source_identity": source_identity,
                    "attack_type": attack_type
                })

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]

        cap = cv2.VideoCapture(item["video_path"])
        cap.set(cv2.CAP_PROP_POS_FRAMES, item["frame_idx"])
        ret, frame = cap.read()
        cap.release()

        if not ret:
            frame = np.zeros((224, 224, 3), dtype=np.uint8)
        else:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        img = Image.fromarray(frame)

        if self.transform:
            img = self.transform(img)

        return (
            img,
            item["label"],
            item["video_path"],
            item["participant_id"],
            item["source_identity"],
            item["attack_type"]
        )

transforms

In [6]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

rebuild validation loader

In [7]:
gc.collect()
torch.cuda.empty_cache()

val_dataset = LazyFaceFrameDataset(val_df, transform=val_transform, num_frames=4)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0)

print("Val frames:", len(val_dataset))

Val frames: 256


recreate model architecture and load

In [8]:
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

best_model_path = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Video/face_binary_resnet18_best.pth"

model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

print("Model loaded successfully.")

Model loaded successfully.


recreate validation function

In [9]:
criterion = nn.CrossEntropyLoss()

def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for images, labels, _, _, _, _ in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = outputs.argmax(dim=1)

            running_loss += loss.item() * images.size(0)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    auc = roc_auc_score(all_labels, all_probs)
    return running_loss / total, correct / total, auc, all_labels, all_preds, all_probs

run frame level evaluation

In [10]:
val_loss, val_acc, val_auc, all_labels, all_preds, all_probs = validate_one_epoch(
    model, val_loader, criterion, device
)

print("\nFRAME-LEVEL RESULTS")
print("Loss:", val_loss)
print("Accuracy:", val_acc)
print("ROC-AUC:", val_auc)
print("\nClassification Report:")
print(classification_report(all_labels, all_preds))
print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))


FRAME-LEVEL RESULTS
Loss: 0.3060152494542763
Accuracy: 0.859375
ROC-AUC: 0.9345914502164502

Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.95      0.82        88
           1       0.97      0.81      0.88       168

    accuracy                           0.86       256
   macro avg       0.85      0.88      0.85       256
weighted avg       0.89      0.86      0.86       256


Confusion Matrix:
[[ 84   4]
 [ 32 136]]


video level inference

In [11]:
def sample_video_frames(video_path, num_frames=4):
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        return []

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return []

    frame_indices = np.linspace(
        0, total_frames - 1,
        num=min(num_frames, total_frames),
        dtype=int
    )

    frames = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret:
            continue
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)

    cap.release()
    return frames

def get_video_level_predictions(model, meta_df, transform, device, num_frames=4):
    model.eval()
    rows = []

    with torch.no_grad():
        for _, row in tqdm(meta_df.iterrows(), total=len(meta_df), desc="Video-level inference"):
            video_path = row["file_path"]
            label = int(row["label"])
            participant_id = row.get("participant_id", None)
            source_identity = row.get("source_identity", None)
            attack_type = row.get("attack_type", None)

            frames = sample_video_frames(video_path, num_frames=num_frames)
            if len(frames) == 0:
                continue

            probs = []
            for frame in frames:
                img = Image.fromarray(frame)
                img = transform(img).unsqueeze(0).to(device)

                outputs = model(img)
                prob_spoof = torch.softmax(outputs, dim=1)[:, 1].item()
                probs.append(prob_spoof)

            mean_prob = float(np.mean(probs))
            pred = 1 if mean_prob >= 0.5 else 0

            rows.append({
                "participant_id": participant_id,
                "source_identity": source_identity,
                "attack_type": attack_type,
                "file_path": video_path,
                "true_label": label,
                "face_pred": pred,
                "face_prob_spoof": mean_prob
            })

    return pd.DataFrame(rows)

run video level evaluation

In [12]:
video_val_results = get_video_level_predictions(
    model=model,
    meta_df=val_df,
    transform=val_transform,
    device=device,
    num_frames=4
)

video_acc = accuracy_score(video_val_results["true_label"], video_val_results["face_pred"])
video_auc = roc_auc_score(video_val_results["true_label"], video_val_results["face_prob_spoof"])

print("\nVIDEO-LEVEL RESULTS")
print("Accuracy:", video_acc)
print("ROC-AUC:", video_auc)
print("\nClassification Report:")
print(classification_report(video_val_results["true_label"], video_val_results["face_pred"]))
print("\nConfusion Matrix:")
print(confusion_matrix(video_val_results["true_label"], video_val_results["face_pred"]))

Video-level inference: 100%|██████████| 66/66 [05:06<00:00,  4.64s/it]


VIDEO-LEVEL RESULTS
Accuracy: 0.90625
ROC-AUC: 0.9718614718614718

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.95      0.88        22
           1       0.97      0.88      0.93        42

    accuracy                           0.91        64
   macro avg       0.89      0.92      0.90        64
weighted avg       0.92      0.91      0.91        64


Confusion Matrix:
[[21  1]
 [ 5 37]]


save fusion-ready predictions

In [13]:
save_preds_path = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Video/face_val_video_predictions.csv"
video_val_results.to_csv(save_preds_path, index=False)

print("Saved:", save_preds_path)

Saved: /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Video/face_val_video_predictions.csv
